In [37]:
import numpy as np
import math
from statistics import mode

In [38]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
data_arr=np.array(data) # Converting the given data into an array
X=data_arr[:,:3].astype(float) # extracting features
y_arr=data_arr[:,3] # extracting labels
y=[]
# One-hot-encoding :- but I mostly didn't use it while training
for i in y_arr:
  if i=='Wine':
    y.append('100')
  elif i=='Beer':
    y.append('010')
  else:
    y.append('001')
y=np.array(y)
max_use=0

In [49]:
# Gini loss function
def gini (lis):
  p=0
  se=np.unique(lis[:,-1])
  for i in se:
    p+=(np.count_nonzero(lis[:,-1]==i)/lis.shape[0])**2
  return (1-p)

# Weighted gini - trying to minimize it i.e., directly maximizing the Information gain
def weighted_gini(groups):
  s=0
  total=float(sum([i.shape[0] for i in groups]))
  for i in groups:
    if i.shape[0] == 0:
        continue
    err=gini(i)
    s+=(i.shape[0])*err/total
  return s

# Entropy loss
def entropy(lis):
  p=0
  se=np.unique(lis[:,-1])
  for i in se:
    p1=(np.count_nonzero(lis[:,-1]==i)/lis.shape[0])
    p+=p1*math.log2(p1)
  return -1*p

# Weighted Entropy loss
def weighted_entropy(groups):
  s=0
  total=float(sum([i.shape[0] for i in groups]))
  for i in groups:
    if i.shape[0] == 0:
        continue
    err=entropy(i)
    s+=(i.shape[0])*err/total
  return s


# Function to do the split according to the values and return the arrays
def test_split(df,value,index):
  left=np.empty((0,4))
  right=np.empty((0,4))
  for rows in df :
    if float(rows[index])<float(value):
      left=np.append(left,[rows],0)
    else:
      right=np.append(right,[rows],0)
  return left,right

# Function to find the best split possible
def best_split_find(df):
  b_err=2
  b_index=None
  b_grps=None
  b_value=None

  if df.shape[0] == 0: # A condition used to stop the split if there is no more split to be done like if we already got a leaf node on left side but not on right
      return None

  for index in range(df.shape[1]-1):
    for rows in df :
      value_to_split = float(rows[index]) # splitting value (threshold)
      groups=test_split(df,value_to_split,index) # calling split function
      # err=weighted_gini(groups) # calculating weighted gini
      err=weighted_entropy(groups) # calculating weighted entropy
      if err == 0 :
        b_err=err
        b_index=index
        b_grps=groups
        b_value=value_to_split
        return {'index':b_index,'value':b_value,'left':b_grps[0],'right':b_grps[1]} # returning the group if we got a perfect split without checking the others
      elif (err < b_err) : # Checking all the values till we get a perfect split or return the best_split with least weighted gini
        b_err=err
        b_index=index
        b_grps=groups
        b_value=value_to_split

  return {'index':b_index,'value':b_value,'left':b_grps[0],'right':b_grps[1]}

In [50]:
# Class Node which has all the attributes that should be stored
class Node :
  def __init__(self,idx,left,right,value):
    self.feature_index=idx
    self.left=left
    self.right=right
    self.value=value

# Recursive logic to build the tree and max_use is the " max depth " parameter just a change in the name
def build_tree(data_arr,name,max_use):
  if max_use==2:
    return name
  else:
    tree=best_split_find(data_arr)
    max_use+=1
    name=Node(tree['index'],tree['left'],tree['right'],tree['value'])
    name.left , name.right=build_tree(name.left,name.left,max_use) , build_tree(name.right,name.right,max_use)
    return name



In [51]:
# decision tree / training the model
naam=None # just a variable
des_tree=build_tree(data_arr,naam,max_use)

In [52]:
# Function used to predict the data using the decision tree made
def predict(X,tree,max_depth):
  if max_depth==2:
    return mode(tree[:,-1])
  else:
    if X[tree.feature_index]<tree.value:
      max_depth+=1
      return predict (X,tree.left,max_depth)
    else:
      max_depth+=1
      return predict(X,tree.right,max_depth)

# Evaluating the model using the training data
predictions=[predict(i,des_tree,0) for i in X]
print(predictions)

# Accuracy
acc= sum(predictions==y_arr)/len(predictions)
print(acc)

[np.str_('Wine'), np.str_('Beer'), np.str_('Whiskey'), np.str_('Wine'), np.str_('Beer'), np.str_('Whiskey'), np.str_('Wine'), np.str_('Beer')]
1.0


In [53]:
# Predicting the labels for test data
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])
pre=[predict(i,des_tree,0) for i in test_data]
print(pre)

[np.str_('Beer'), np.str_('Whiskey'), np.str_('Wine')]
